Untuk dataset sidik jari **SOCOFing** (`ruizgara/socofing`), pendekatan paling optimal adalah menggunakan input **Grayscale 1-channel (`1 × 128 × 128`)**. Selain karena citra sidik jari pada dasarnya hitam-putih, 1-channel menghemat memori buffer input hingga 3× lipat saat disintesis ke HLS.

Dataset SOCOFing berisi dua kategori utama: **`Real`** (6.000 sidik jari asli) dan **`Altered`** (~49.000 sidik jari yang dimanipulasi dengan *obliteration*, *central rotation*, dan *Z-cut*). Di skrip berikut, jumlah kedua kelas otomatis diseimbangkan (masing-masing **1.500 foto = 3.000 total**) agar proses *training* di Colab cepat dan akurasinya tidak bias.

---

### Langkah 1: Unduh & Siapkan Dataset SOCOFing di Colab

Jalankan sel ini terlebih dahulu. Menggunakan `kagglehub`, dataset langsung terunduh ke Colab tanpa perlu mengunggah file `kaggle.json`:

In [1]:
import os
import glob
import random
import kagglehub

# 1. Download dataset SOCOFing langsung dari Kaggle
dataset_dir = kagglehub.dataset_download("ruizgara/socofing")
print("Lokasi download:", dataset_dir)

# 2. Cari otomatis folder Real dan Altered
real_imgs = glob.glob(os.path.join(dataset_dir, "**", "Real", "*.BMP"), recursive=True)
altered_imgs = glob.glob(os.path.join(dataset_dir, "**", "Altered", "**", "*.BMP"), recursive=True)

# Ambil sampel seimbang (1.500 Real vs 1.500 Altered) agar training cepat & adil
random.seed(42)
real_sample = random.sample(real_imgs, min(1500, len(real_imgs)))
altered_sample = random.sample(altered_imgs, min(1500, len(altered_imgs)))

all_paths = real_sample + altered_sample
all_labels = [0] * len(real_sample) + [1] * len(altered_sample)  # 0: Real, 1: Altered
class_names = ["Real", "Altered"]

print(f"Total digunakan: {len(all_paths)} foto ({len(real_sample)} Real, {len(altered_sample)} Altered)")

Using Colab cache for faster access to the 'socofing' dataset.
Lokasi download: /kaggle/input/socofing
Total digunakan: 3000 foto (1500 Real, 1500 Altered)


---

### Langkah 2: Arsitektur `LSI_CNN_128x128` + Data Loader

Pada resolusi **128×128**, ditambahkan 1 blok konvolusi (`conv4` + `pool4`) agar dimensi *flatten* tetap **`64 × 4 × 4 = 1024`**. Dengan cara ini:

* Model punya 4 tahap ekstraksi fitur untuk menangkap garis alur (*minutiae*) dan bekas potongan manipulasi sidik jari.
* Ukuran *fully connected* (`fc1`) tetap ramping (~131 KB), sehingga total seluruh bobot model hanya **~191 KB** (aman untuk *on-chip* BRAM FPGA).



*(Catatan: `RandomHorizontalFlip` sengaja tidak dipakai karena membalik sidik jari secara horizontal merusak arah alami alur sidik jari).*

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device aktif:", device)

# 1. Kuantisasi Power-of-Two (PoT) dengan STE
class PoTQuantize(torch.autograd.Function):
    @staticmethod
    def forward(ctx, weight):
        sign = torch.sign(weight)
        abs_weight = torch.clamp(torch.abs(weight), min=1e-5)
        log2_weight = torch.round(torch.log2(abs_weight))
        return sign * (2.0 ** log2_weight)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output

def quantize_pot(weight):
    return PoTQuantize.apply(weight)

# 2. Arsitektur CNN 128x128 (1-Channel Grayscale)
class LSI_CNN_128x128(nn.Module):
    def __init__(self, num_classes=2):
        super(LSI_CNN_128x128, self).__init__()
        self.qat_mode = False

        # Input: 1 x 128 x 128 -> Conv1(stride=2) + Pool1 -> 16 x 32 x 32
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)

        # 16 x 32 x 32 -> Conv2 + Pool2 -> 32 x 16 x 16
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)

        # 32 x 16 x 16 -> Conv3 + Pool3 -> 64 x 8 x 8
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)

        # 64 x 8 x 8 -> Conv4 + Pool4 -> 64 x 4 x 4 (Flatten = 1024)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1)
        self.pool4 = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 4 * 4, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        if self.qat_mode:
            x = F.relu(self.pool1(F.conv2d(x, quantize_pot(self.conv1.weight), self.conv1.bias, stride=2, padding=1)))
            x = F.relu(self.pool2(F.conv2d(x, quantize_pot(self.conv2.weight), self.conv2.bias, stride=1, padding=1)))
            x = F.relu(self.pool3(F.conv2d(x, quantize_pot(self.conv3.weight), self.conv3.bias, stride=1, padding=1)))
            x = F.relu(self.pool4(F.conv2d(x, quantize_pot(self.conv4.weight), self.conv4.bias, stride=1, padding=1)))

            x = x.view(x.size(0), -1)
            x = F.relu(F.linear(x, quantize_pot(self.fc1.weight), self.fc1.bias))
            x = self.dropout(x)
            x = F.linear(x, quantize_pot(self.fc2.weight), self.fc2.bias)
        else:
            x = F.relu(self.pool1(self.conv1(x)))
            x = F.relu(self.pool2(self.conv2(x)))
            x = F.relu(self.pool3(self.conv3(x)))
            x = F.relu(self.pool4(self.conv4(x)))

            x = x.view(x.size(0), -1)
            x = F.relu(self.fc1(x))
            x = self.dropout(x)
            x = self.fc2(x)
        return x

# 3. Dataset & Augmentasi Khusus Sidik Jari (128x128 Grayscale)
transform_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 128)),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor()
])

transform_val = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

class FingerprintDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("L")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

# Bagi 80% Train (2.400 foto) : 20% Test/Val (600 foto) secara seimbang
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels, test_size=0.2, stratify=all_labels, random_state=42
)

dataset_train = FingerprintDataset(train_paths, train_labels, transform=transform_train)
dataset_val   = FingerprintDataset(val_paths, val_labels, transform=transform_val)

data_loader = DataLoader(dataset_train, batch_size=32, shuffle=True)
data_loader_test = DataLoader(dataset_val, batch_size=32, shuffle=False)

model = LSI_CNN_128x128(num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
print(f"Siap! Train: {len(dataset_train)} | Val: {len(dataset_val)}")

Device aktif: cpu
Siap! Train: 2400 | Val: 600


---

### Langkah 3: Training Tahap 1 (Float32) & Tahap 2 (QAT Power-of-Two)

Jalankan sel ini untuk melatih model sampai konvergen lalu mengunci bobotnya ke pangkat dua ($\pm 2^k$):

In [3]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100.0 * correct / total

# --- TAHAP 1: FLOAT32 TRAINING ---
print("=== TAHAP 1: STANDARD TRAINING (FLOAT32) ===")
model.qat_mode = False
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(25):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in data_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    if (epoch + 1) % 5 == 0 or epoch == 0:
        val_acc = evaluate(model, data_loader_test)
        print(f"Epoch {epoch+1:02d}/25 | Loss: {total_loss/len(data_loader):.4f} | "
              f"Train Acc: {100*correct/total:.2f}% | Val Acc: {val_acc:.2f}%")

# --- TAHAP 2: QAT POWER-OF-TWO FINE-TUNING ---
print("\n=== TAHAP 2: QAT POWER-OF-TWO ===")
model.qat_mode = True
optimizer_qat = optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(10):
    model.train()
    total_loss = 0.0
    for imgs, labels in data_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_qat.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_qat.step()
        total_loss += loss.item()

    val_acc_qat = evaluate(model, data_loader_test)
    print(f"QAT Epoch {epoch+1:02d}/10 | Loss: {total_loss/len(data_loader):.4f} | Val Acc (PoT): {val_acc_qat:.2f}%")

torch.save(model.state_dict(), "model_sidik_jari_128x128_pot.pth")

=== TAHAP 1: STANDARD TRAINING (FLOAT32) ===
Epoch 01/25 | Loss: 0.6944 | Train Acc: 50.08% | Val Acc: 50.00%
Epoch 05/25 | Loss: 0.5273 | Train Acc: 74.46% | Val Acc: 84.67%
Epoch 10/25 | Loss: 0.2763 | Train Acc: 89.38% | Val Acc: 87.00%
Epoch 15/25 | Loss: 0.2283 | Train Acc: 91.46% | Val Acc: 91.83%
Epoch 20/25 | Loss: 0.1895 | Train Acc: 93.17% | Val Acc: 93.17%
Epoch 25/25 | Loss: 0.1658 | Train Acc: 93.96% | Val Acc: 95.83%

=== TAHAP 2: QAT POWER-OF-TWO ===
QAT Epoch 01/10 | Loss: 0.1939 | Val Acc (PoT): 94.83%
QAT Epoch 02/10 | Loss: 0.1510 | Val Acc (PoT): 95.33%
QAT Epoch 03/10 | Loss: 0.1600 | Val Acc (PoT): 95.83%
QAT Epoch 04/10 | Loss: 0.1555 | Val Acc (PoT): 94.33%
QAT Epoch 05/10 | Loss: 0.1593 | Val Acc (PoT): 94.33%
QAT Epoch 06/10 | Loss: 0.1544 | Val Acc (PoT): 95.17%
QAT Epoch 07/10 | Loss: 0.1518 | Val Acc (PoT): 95.33%
QAT Epoch 08/10 | Loss: 0.1453 | Val Acc (PoT): 94.50%
QAT Epoch 09/10 | Loss: 0.1501 | Val Acc (PoT): 95.83%
QAT Epoch 10/10 | Loss: 0.1290 | Va

---

### Penyesuaian Singkat untuk Ekspor ke `weights.h` & C++ HLS

Jika hasil akurasi QAT-nya sudah memuaskan dan ingin dipindahkan ke C++ HLS seperti skrip sebelumnya, hanya ada **3 penyesuaian kecil**:

1. **Di skrip `export_hls.py`:** Tambahkan `("conv4", model.conv4)` ke dalam daftar `layers`.
2. **Di `cnn_hls.cpp`:**
* Ubah ukuran buffer ping-pong menjadi `static int32_t buf_a[65536]; static int32_t buf_b[65536];` (menampung output `conv1` sebesar `16 × 64 × 64 = 65.536`).
* Ubah input fungsi utama menjadi `const uint8_t img[1][128][128]` (1 channel) dan panggil `conv1` dengan `in_c = 1, in_h = 128, in_w = 128`.
* Tambahkan pemanggilan `conv2d_relu` + `maxpool2x2` untuk `conv4` (`64 → 64`, `8×8 → 4×4`) sebelum masuk ke `linear_layer` (`fc1`).

In [4]:
import numpy as np
from google.colab import files

SCALE_BITS = 12
SCALE_FACTOR = 1 << SCALE_BITS  # Q4.12 (1.0 = 4096)

def encode_pot_weights(weight_tensor):
    w_pot = quantize_pot(weight_tensor).detach().cpu().numpy().reshape(-1)
    encoded = []
    for w in w_pot:
        if w == 0.0:
            encoded.append(0)
            continue
        sign = 1 if w > 0 else -1
        k = int(round(np.log2(abs(w))))
        if k < -13:
            encoded.append(0)
        else:
            k = max(-13, min(15, k))
            encoded.append(sign * (k + 16))
    return np.array(encoded, dtype=np.int8)

def encode_bias(bias_tensor):
    b = bias_tensor.detach().cpu().numpy().reshape(-1)
    return np.round(b * SCALE_FACTOR).astype(np.int32)

def format_c_array(arr, name, c_type, per_line=16):
    lines = [f"const {c_type} {name}[{len(arr)}] = {{"]
    for i in range(0, len(arr), per_line):
        chunk = ", ".join(str(int(x)) for x in arr[i:i+per_line])
        lines.append(f"    {chunk},")
    lines.append("};\n")
    return "\n".join(lines)

layers = [
    ("conv1", model.conv1),
    ("conv2", model.conv2),
    ("conv3", model.conv3),
    ("conv4", model.conv4),
    ("fc1",   model.fc1),
    ("fc2",   model.fc2),
]

num_classes = model.fc2.out_features

with open("weights.h", "w") as f:
    f.write("#ifndef WEIGHTS_H\n#define WEIGHTS_H\n\n")
    f.write("#include <stdint.h>\n\n")
    f.write(f"#define NUM_CLASSES {num_classes}\n")
    f.write(f"#define FRAC_BITS {SCALE_BITS}\n\n")

    for name, layer in layers:
        w_enc = encode_pot_weights(layer.weight)
        b_enc = encode_bias(layer.bias)
        f.write(format_c_array(w_enc, f"{name}_weight", "int8_t"))
        f.write(format_c_array(b_enc, f"{name}_bias", "int32_t"))

    f.write("#endif // WEIGHTS_H\n")

print("File weights.h (128x128) berhasil dibuat!")
files.download("weights.h")

File weights.h (128x128) berhasil dibuat!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>